# Divya - RLCD fine-tune of System-1 (Laya) on Indian corporate filings

Kaggle GPU **T4 x2** - the exact hardware Laya's own reference recipe uses. All heavy
compute happens here; the local machine only writes code and scrapes small JSON.

**What this decides.** `docs/loop/STOP_RULE.md` pre-commits, before any training:

> If a fine-tuned Laya does not reach **>= 0.65 macro-F1 on the 120 held-out NSE
> announcements**, the typed-decision approach is refuted for this domain.

The 120 are excluded from the corpus **by id** and evaluated exactly once, at the end.

**What leaves this kernel.** Only the sparse delta, the results, and the small manifests.
The 1.7 GB checkpoint does not: `kaggle kernels output` returns it as **0 bytes**,
silently. See the last cell.

In [ ]:
import os, sys, json, time, glob, shutil, traceback, contextlib, textwrap

!pip install -q laya transformers safetensors huggingface_hub 2>&1 | tail -1

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  gpu{i}: {torch.cuda.get_device_name(i)} '
              f'{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GB')
else:
    print('  NO GPU -- this kernel must run with GPU T4 x2 enabled')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


# --- failure reporting ------------------------------------------------------------
#
# `kaggle kernels output` serves NOTHING for a kernel that errors, and this account's
# token cannot read the web-side kernel log. So a crash is indistinguishable from a silent
# hang. Everything below therefore appends to a file in /kaggle/working, which does get
# served: a failed run must still deliver its own traceback.
RUNLOG = '/kaggle/working/RUNLOG.txt'

def log(msg):
    with open(RUNLOG, 'a') as f:
        f.write(str(msg) + '\n')
    print(msg, flush=True)

@contextlib.contextmanager
def guard(name):
    try:
        yield
    except BaseException:
        tb = traceback.format_exc()
        log(f'\n### FAILED CELL: {name}\n{tb}')
        raise

log(f'=== kernel start {time.strftime("%Y-%m-%d %H:%M:%S")} torch={torch.__version__} '
    f'device={DEVICE} ===')


In [ ]:
# RUNLOG note: this cell is guarded
import textwrap as _tw
_src = _tw.dedent('def find_dir(filename):\n    """Resolve a file\'s directory by CONTENT, not by path pattern.\n\n    Kaggle creates several directories under /kaggle/input -- including a bare \'datasets\'\n    owner folder -- and a glob for the dataset name matched the wrong one. Asking which\n    directory actually holds the corpus cannot be wrong.\n    """\n    roots = [d for d in glob.glob(\'/kaggle/input/*\') if os.path.isdir(d)]\n    roots += [os.getcwd(), \'/kaggle/working\']\n    for r in roots:\n        if os.path.isfile(os.path.join(r, filename)):\n            return r\n    for r in roots:\n        for dirpath, _dirs, files in os.walk(r):\n            if filename in files:\n                return dirpath\n    raise SystemExit(f\'could not find {filename} under /kaggle/input\')\n\nDATASET_DIR = find_dir(\'finetune_corpus.jsonl\')\nprint(\'DATASET_DIR =\', DATASET_DIR)\nprint(sorted(os.listdir(DATASET_DIR)))\n')
with guard("cell-2"):
    exec(compile(_src, "<cell-2>", "exec"), globals())


## Zero-shot baseline on dev, for the record

In [ ]:
# RUNLOG note: this cell is guarded
import textwrap as _tw
_src = _tw.dedent("CORPUS = os.path.join(DATASET_DIR, 'finetune_corpus.jsonl')\nHELDOUT = os.path.join(DATASET_DIR, 'nse_v5_eval_set.jsonl')\nTRAINER = '/kaggle/working/finetune_laya.py'\n\nbase = (f'python {TRAINER} --eval-only --limit-dev 200 --device {DEVICE} '\n        f'--corpus {CORPUS} --heldout {HELDOUT}')\nprint(base)\n!{base}\n")
with guard("cell-4"):
    exec(compile(_src, "<cell-4>", "exec"), globals())


## The fine-tune

Single process with gradient accumulation rather than the reference's DDP. Two T4s are two
cards; DDP buys throughput, not quality. Micro-batch 8 x accum 8 gives an effective batch
of **64**, matching the reference.

`--delta-out` writes the sparse delta **from inside the training process**, because the base
state dict is still in memory there. An earlier version downloaded the base checkpoint a
second time at the end to build the delta; that 1.7 GB re-download is the most fragile step
in the kernel and it is now gone.

In [ ]:
# RUNLOG note: this cell is guarded
import textwrap as _tw
_src = _tw.dedent("train_cmd = (\n    f'python {TRAINER} --epochs 4 --micro-batch 8 --accum 8 --group-size 4 '\n    f'--device {DEVICE} --corpus {CORPUS} --heldout {HELDOUT} '\n    f'--limit-dev 200 --out /kaggle/working/finetuned '\n    f'--delta-out /kaggle/working/delta --delta-keep 0.03'\n)\nprint(train_cmd)\nt0 = time.time()\n!{train_cmd}\nprint(f'\\nTRAINING WALL TIME: {time.time()-t0:.0f}s')\nprint('model saved:', os.path.isfile('/kaggle/working/finetuned/model.safetensors'))\nprint('delta saved:', os.path.isfile('/kaggle/working/delta/delta.safetensors'))\n")
with guard("cell-6"):
    exec(compile(_src, "<cell-6>", "exec"), globals())


## Evaluate ONCE on the 120 held-out announcements

This is the number that decides the project. Computed once. If it misses the bar the stop
rule fires and this set is not tuned against again.

`fp = pred[lb] - tp[lb]`. An earlier version computed `fp = (N - tot[lb]) - tp`, which
yields a *negative* false-positive count and a precision above 1 -- it reported macro-F1
0.2786 sitting next to an accuracy of 0.975, which would have fired the stop rule on a
false ground. The confusion matrix is printed in full so the metric can be re-derived.

In [ ]:
# RUNLOG note: this cell is guarded
import textwrap as _tw
_src = _tw.dedent('import json\nfrom collections import Counter\nfrom safetensors.torch import load_file\nimport torch\nfrom laya.agent import _load_tokenizer\nfrom laya.common import QTYPES, build_sequence, render_options\n\nOUT = \'/kaggle/working/finetuned\'\nif not os.path.isfile(os.path.join(OUT, \'model.safetensors\')):\n    raise SystemExit(\'no fine-tuned weights -- the training cell did not save\')\n\ncfg = json.load(open(os.path.join(OUT, \'rl_agent_config.json\')))\ntok = _load_tokenizer(os.path.join(OUT, \'tokenizer\'), cfg)\nmodel = build_model(cfg, encoder_dir=os.path.join(OUT, \'encoder\'))\nmodel.load_state_dict(load_file(os.path.join(OUT, \'model.safetensors\')), strict=True)\nmodel.to(DEVICE).eval()\n\nrows = [json.loads(l) for l in open(HELDOUT) if l.strip()]\noptions = sorted({r[\'labels\'][\'event_type\'] for r in rows})\nq = {\'t\': \'choice\', \'ins\': \'Classify the corporate event this filing discloses.\',\n     \'crit\': {o: \'\' for o in options}}\nk = len(render_options(q))\nprint(\'held-out n =\', len(rows), \'| classes =\', options)\n\nconf = Counter()          # (gold, predicted) -> count. This is what macro-F1 actually needs.\nskipped = 0\nwith torch.no_grad():\n    for r in rows:\n        seq, markers = build_sequence(tok, r[\'content\'], q, cfg[\'max_len\'], cfg[\'head_max_len\'])\n        if len(markers) != k:\n            skipped += 1\n            continue\n        ids = torch.tensor([seq], device=DEVICE); att = torch.ones_like(ids)\n        mpos = torch.tensor([markers], device=DEVICE)\n        mmask = torch.ones_like(mpos, dtype=torch.bool)\n        qt = torch.tensor([QTYPES[\'choice\']], device=DEVICE)\n        logits, _ = model(ids, att, mpos, mmask, qt)\n        conf[(r[\'labels\'][\'event_type\'], options[int(logits.float().reshape(-1).argmax())])] += 1\n\ntot, pred, ok = Counter(), Counter(), Counter()\nfor (g, p), c in conf.items():\n    tot[g] += c; pred[p] += c; ok[g] += c if g == p else 0\n\nf1s = []\nfor lb in options:\n    tp = ok[lb]; fn = tot[lb] - tp; fp = pred[lb] - tp      # NOT (N - tot[lb]) - tp\n    assert fp >= 0, f\'negative false positive for {lb}: {fp}\'\n    p = tp / (tp + fp) if tp + fp else 0.0\n    rc = tp / (tp + fn) if tp + fn else 0.0\n    f1s.append(2 * p * rc / (p + rc) if p + rc else 0.0)\n\nn = max(sum(tot.values()), 1)\nBAR = 0.65\nres = {\'n\': n, \'skipped\': skipped,\n       \'accuracy\': round(sum(ok.values()) / n, 4),\n       \'macro_f1\': round(sum(f1s) / len(f1s), 4) if f1s else 0.0,\n       \'bar\': BAR, \'device\': DEVICE,\n       \'confusion\': {f\'{g}->{p}\': c for (g, p), c in sorted(conf.items())},\n       \'per_class\': {lb: round(ok[lb] / tot[lb], 4) for lb in options if tot[lb]}}\nres[\'verdict\'] = \'PASSED\' if res[\'macro_f1\'] >= BAR else \'FIRED\'\n\nprint(f"HELD-OUT  n={n}  skipped={skipped}  accuracy={res[\'accuracy\']}  macro_f1={res[\'macro_f1\']}")\nfor lb in options:\n    if tot[lb]:\n        print(f\'  {lb:<20} n={tot[lb]:<3} acc={ok[lb]/tot[lb]:.3f}\')\nprint(\'CONFUSION\', json.dumps(res[\'confusion\']))\nprint(f"STOP RULE (>= {BAR} macro-F1 on held-out): {res[\'verdict\']}")\njson.dump(res, open(\'/kaggle/working/HELD_OUT_RESULT.json\', \'w\'), indent=2)\nprint(\'wrote HELD_OUT_RESULT.json\')\n')
with guard("cell-8"):
    exec(compile(_src, "<cell-8>", "exec"), globals())


## What leaves, and what stays

`kaggle kernels output` streams the **whole** working directory. The working directory
contains a 1.7 GB safetensors, so the transfer is never small -- it is why the weights
arrived as 0 bytes. The delta is the only model artifact that needs to travel, and it is
reconstructible against the stock checkpoint, which downloads from HF without trouble.

Nothing is deleted before `HELD_OUT_RESULT.json` and the delta are on disk and their
sizes have been printed. This cell cannot fail the run: everything in it is guarded.

In [ ]:
# RUNLOG note: this cell is guarded
import textwrap as _tw
_src = _tw.dedent('def human(n):\n    for u in (\'B\', \'KB\', \'MB\', \'GB\'):\n        if n < 1024 or u == \'GB\':\n            return f\'{n:.1f} {u}\'\n        n /= 1024\n\ndef report(root, label):\n    total = 0\n    for dirpath, _d, files in os.walk(root):\n        for f in files:\n            p = os.path.join(dirpath, f)\n            total += os.path.getsize(p)\n    print(f\'{label}: {human(total)}\')\n    return total\n\ntry:\n    # prove the two small artifacts are really on disk before anything is removed\n    for p in (\'/kaggle/working/HELD_OUT_RESULT.json\',\n              \'/kaggle/working/delta/delta.safetensors\',\n              \'/kaggle/working/delta/delta_index.json\',\n              \'/kaggle/working/finetuned/divya_finetune_meta.json\'):\n        ok = os.path.isfile(p)\n        print((\'OK   \' if ok else \'MISS \'), p, human(os.path.getsize(p)) if ok else \'\')\n\n    report(\'/kaggle/working/finetuned\', \'BEFORE  finetuned (not shipped)\')\n    report(\'/kaggle/working/delta\',    \'BEFORE  delta    (shipped)   \')\n\n    shutil.rmtree(\'/kaggle/working/finetuned\', ignore_errors=True)\n    # the base checkpoint lives in the HF cache; the delta is applied against it locally\n    shutil.rmtree(os.path.expanduser(\'~/.cache/huggingface\'), ignore_errors=True)\n    torch.cuda.empty_cache()\nexcept Exception:\n    traceback.print_exc()\n\nprint(\'\\nFINAL /kaggle/working CONTENTS:\')\nfor dirpath, _d, files in os.walk(\'/kaggle/working\'):\n    for f in sorted(files):\n        p = os.path.join(dirpath, f)\n        print(f\'  {os.path.relpath(p, "/kaggle/working"):40s} {human(os.path.getsize(p)):>10s}\')\n')
with guard("cell-10"):
    exec(compile(_src, "<cell-10>", "exec"), globals())
